# Run LogMap Alignment

Runs `LogMap` and `LogMapBIO` alignment tools across all ontology benchmark subsets, producing RDF mapping files and reduced candidate lists for downstream LLM oracle evaluation.

The first cell invokes `run_logmap_alignment`, executing the standard LogMap matcher via Docker (`amazoncorretto:8-alpine`, 10 GB heap) over all subsets from `anatomy`, `bioml-2024`, `largebio`, and `largebio_small`. Outputs are written to `output/logmap/{dataset}/{subset}/`, producing `logmap_mappings.rdf` and `logmap_mappings_to_ask_oracle_user_llm.txt` for LLM oracle verification.

The second cell mirrors this using `run_logmap_bio`, a biomedical-domain LogMap extension running locally via the `JAVA_EXE` executable from `.env`. Outputs follow the same structure under `output/logmapbio/{dataset}/{subset}/`.

Both wrappers live in `modules/logmap_wrapper.py` and return structured dicts handling subprocess errors, missing outputs, and environment misconfiguration.

\
**Note:** The pipeline automatically stores all intermediate and generated results in the `output/` directory, while finalized and manually curated results are saved in the `final_results/` directory.

### Run standard LogMap matcher

In [ ]:
from modules.logmap_wrapper import run_logmap_alignment
from utils.utils import format_subsets_ontologies_paths
from pathlib import Path
import os

ALL_DATASET_NAMES = {
    "anatomy": ["human-mouse"],
    "bioml-2024": ["snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm", "ncit-doid", "omim-ordo"],
    "largebio": ["fma-snomed", "snomed-nci", "fma-nci"],
    "largebio_small": ["fma-nci", "snomed-nci", "fma-nci"],
}

# Run LogMap alignment for all datasets
workspace_root = os.getenv("WORKSPACE_ROOT")

for dataset_name, subfolders in ALL_DATASET_NAMES.items():
    for subfolder in subfolders:
        o1_path, o2_path = format_subsets_ontologies_paths(dataset_name, subfolder)
        output_dir = f"output/logmap/{dataset_name}/{subfolder}"
        
        # Create output directory
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        
        result = run_logmap_alignment(
            o1_path=o1_path,
            o2_path=o2_path,
            output_dir=output_dir,
            workspace_root=workspace_root
        )
        print(f"Processed {dataset_name}/{subfolder}: {result}")

### Run LogMapBIO to mediate biomedical ontologies

In [ ]:
from modules.logmap_wrapper import run_logmap_bio
from utils.utils import format_subsets_ontologies_paths
from pathlib import Path
import os

ALL_DATASET_NAMES = {
    "anatomy": ["human-mouse"],
    "bioml-2024": ["snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm", "ncit-doid", "omim-ordo"],
    "largebio": ["fma-snomed", "snomed-nci", "fma-nci"],
    "largebio_small": ["fma-nci", "snomed-nci", "fma-nci"],
}

workspace_root = os.getenv("WORKSPACE_ROOT")

for dataset_name, subfolders in ALL_DATASET_NAMES.items():
    for subfolder in subfolders:

        o1_path, o2_path = format_subsets_ontologies_paths(dataset_name, subfolder)
        output_dir = f"output/logmapbio/{dataset_name}/{subfolder}"
        
        # Create output directory
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        
        result = run_logmap_bio(
            o1_path=o1_path,
            o2_path=o2_path,
            output_dir=output_dir,
            workspace_root=workspace_root,
            java_heap="12g",
        )
        
        print(f"Processed {dataset_name}/{subfolder}: {result}")